In [ ]:
# 1. Xử lý giá trị thiếu cho BMI
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

In [ ]:
# 2. Loại bỏ cột ID
df.drop(['id'], axis=1, inplace=True)

In [ ]:
# 3. Mã hóa biến phân loại (Categorical to Numerical)
# Chuyển đổi các cột chữ thành các cột số 0 và 1
df = pd.get_dummies(df, columns=['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status'], drop_first=True)

In [ ]:
# 4. Tách đặc trưng (X) và nhãn (y)
X = df.drop('stroke', axis=1)
y = df['stroke']

In [ ]:
# 5. Chia tập Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 6. Xử lý mất cân bằng bằng SMOTE (Tạo thêm dữ liệu cho nhóm thiểu số)
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [ ]:
# 7. Chuẩn hóa dữ liệu (Bắt buộc cho SVM và Neural Network)
scaler = StandardScaler()
X_train_res = scaler.fit_transform(X_train_res)
X_test = scaler.transform(X_test)

In [ ]:
# 8. Tìm đặc trưng
def fitness_function(individual, X_tr, y_tr, X_te, y_te):
    selected = [i for i, bit in enumerate(individual) if bit == 1]
    if len(selected) == 0: return 0
    # Dùng Decision Tree để đánh giá nhanh "độ tốt" của bộ đặc trưng
    model = DecisionTreeClassifier(max_depth=5, random_state=42)
    model.fit(X_tr[:, selected], y_tr)
    return f1_score(y_te, model.predict(X_te[:, selected]))

def run_ga(X_tr, y_tr, X_te, y_te, n_features):
    # Thiết lập seed trước GA để đảm bảo kết quả không thay đổi
    np.random.seed(42)
    
    pop_size = 15
    generations = 10
    # Khởi tạo quần thể ngẫu nhiên
    population = np.random.randint(0, 2, (pop_size, n_features))
    
    for g in range(generations):
        scores = [fitness_function(ind, X_tr, y_tr, X_te, y_te) for ind in population]
        # Chọn lọc các cá thể tốt nhất
        best_indices = np.argsort(scores)[-pop_size//2:]
        parents = population[best_indices]
        
        # Tạo thế hệ mới qua lai ghép
        offspring = []
        for i in range(pop_size - len(parents)):
            p1, p2 = parents[np.random.randint(0, len(parents), 2)]
            child = np.where(np.random.rand(n_features) > 0.5, p1, p2)
            offspring.append(child)
        
        population = np.vstack([parents, offspring])
        print(f"Gen {g+1}: Best F1 = {max(scores):.4f}")
    
    return population[np.argmax(scores)]

# Chạy GA để chọn đặc trưng
best_mask = run_ga(X_train_res, y_train_res, X_test, y_test, X_train_res.shape[1])
selected_features_idx = [i for i, bit in enumerate(best_mask) if bit == 1]
print(f"Số lượng đặc trưng tối ưu được chọn: {len(selected_features_idx)}")

In [ ]:
# Lọc lại dữ liệu theo GA 
X_train_ga = X_train_res[:, selected_features_idx]
X_test_ga = X_test[:, selected_features_idx]